In [2]:
import torch

graph = torch.load("./dataset/graphs/action-recognition-in-videos-on-something.pt")
print(graph)

Data(x=[0], edge_index=[2, 1565], arxiv_id=[876], content=[876], abstract=[876], title=[876], desc='Merged graph statistics:
Original total nodes before merging: 1792
Number of nodes: 876
Number of edges: 1565

Root papers to build up the graph:
2311.15769v1: Side4Video: Spatial-Temporal Side Network for Memory-Efficient Image-to-Video Transfer Learning
2112.09133v2: Masked Feature Prediction for Self-Supervised Visual Pre-Training
2203.12602v3: VideoMAE: Masked Autoencoders are Data-Efficient Learners for Self-Supervised Video Pre-Training
2207.11660v1: MAR: Masked Autoencoders for Efficient Action Recognition
2212.04500v2: Masked Video Distillation: Rethinking Masked Feature Modeling for Self-supervised Video Representation Learning
2303.12001v2: ViC-MAE: Self-Supervised Representation Learning from Images and Video with Contrastive Masked Autoencoders
2212.03191v2: InternVideo: General Video Foundation Models via Generative and Discriminative Learning
2306.00989v1: Hiera: A Hierarch

/tmp/ipykernel_85955/2372515897.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  graph = torch.load("./dataset/graphs/action-recognition-in-videos-on-something.pt")


In [2]:
from src.utils.citation_graph import to_hierarchical

hier_version = to_hierarchical(graph, 1000)
print(hier_version)

Data(edge_index=[2, 35578], node_text=[31048], root_labels=[31048])


In [3]:
root_indices = torch.where(hier_version.root_labels == 1)[0]
print(root_indices.shape)

torch.Size([847])


In [12]:
def hier_graphs(merged_paper_graph: Data, citation_graph: Data) -> Data:
    """
    Merge hierarchical and citation graphs by adding edges between matching root nodes,
            
    Returns:
        Data: Updated hierarchical graph with new edges from citation network
    """

    root_indices = torch.where(merged_paper_graph.root_labels == 1)[0]

    chunks_to_idx = {merged_paper_graph.node_text[idx]: idx for idx in root_indices}
    cit_title_to_idx = {title: idx for idx, title in enumerate(citation_graph.title)}
    cit_idx_to_title = {idx: title for idx, title in enumerate(citation_graph.title)}

    new_edges = []

    for hier_idx in root_indices:
        hier_title = merged_paper_graph.node_text[hier_idx]

        if hier_title in cit_title_to_idx:
            cit_idx = cit_title_to_idx[hier_title]

            # Get all citations from this paper in citation graph
            citation_edges = citation_graph.edge_index
            source_mask = citation_edges[0] == cit_idx
            cited_indices = citation_edges[1][source_mask]

            for cited_idx in cited_indices:
                cited_title = cit_idx_to_title[cited_idx.item()]
                
                # If the cited paper exists as a root node in hierarchical graph
                if cited_title in chunks_to_idx:
                    cited_hier_idx = chunks_to_idx[cited_title]
                    new_edges.append((hier_idx, cited_hier_idx))

            # Also get papers that cite this paper
            target_mask = citation_edges[1] == cit_idx
            citing_indices = citation_edges[0][target_mask]
            
            # For each citing paper
            for citing_idx in citing_indices:
                citing_title = cit_idx_to_title[citing_idx.item()]
                
                # If the citing paper exists as a root node in hierarchical graph
                if citing_title in chunks_to_idx:
                    citing_hier_idx = chunks_to_idx[citing_title]
                    new_edges.append((citing_hier_idx, hier_idx))

    if new_edges:
        # Convert new edges to tensor
        new_edges_tensor = torch.tensor(new_edges, dtype=torch.long).t()
        
        # Combine with existing edges
        updated_edge_index = torch.cat([merged_paper_graph.edge_index, new_edges_tensor], dim=1)
        
        # Create updated graph
        updated_graph = Data(
            edge_index=updated_edge_index,
            node_text=merged_paper_graph.node_text,
            root_labels=merged_paper_graph.root_labels
        )
    else:
        updated_graph = merged_paper_graph
    
    return updated_graph

    
hier_graph = hier_graphs(merged_graph, graph)
print(hier_graph)



Data(edge_index=[2, 88], node_text=[76], root_labels=[76])


In [16]:
import re
import itertools
from collections import defaultdict
import torch
from torch_geometric.data import Data
import spacy

nlp = spacy.load("en_core_web_sm")

def split_into_chunks(text, chunk_size=100):
    words = text.split()
    chunks = [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]
    return chunks

def build_graph_from_tex_content(content, main_title="Main Title"):
    node_list = []
    edges = []
    section_refs = defaultdict(list)
    section_stack = []  # Stack to manage the hierarchy of sections

    # Add the root node
    root_node = len(node_list)
    node_list.append({"title": main_title, "content": "", "type": "root"})

    # Regex pattern to match any level of section (\section, \subsection, \subsubsection, etc.)
    pattern = r'(\\(sub)*section\{.*?\})'
    parts = re.split(pattern, content)

    current_parent_node = root_node  # Start with root as the initial parent
    for i in range(len(parts)):
        if isinstance(parts[i], str) and re.match(r'\\(sub)*section', parts[i]):
            # Calculate depth based on the number of "sub" prefixes
            depth = parts[i].count("sub")
            title_match = re.match(r'\\(sub)*section\{(.*?)\}', parts[i])
            if title_match:
                section_title = title_match.group(2)

                # Remove nodes from stack that are at the same or deeper level
                while section_stack and section_stack[-1][1] >= depth:
                    section_stack.pop()

                # Create a new intermediate node for this section
                section_node = len(node_list)
                node_list.append({"title": section_title, "content": "", "type": f"section_level_{depth}"})

                # Determine the parent node
                if section_stack:
                    parent_node = section_stack[-1][0]
                else:
                    parent_node = root_node  # Connect top-level sections to the root

                # Connect the current section to its parent
                edges.append((parent_node, section_node))

                # Push the current section onto the stack with its depth level
                section_stack.append((section_node, depth))

                # If there is content after the section title, split it into chunks
                if i + 1 < len(parts) and isinstance(parts[i + 1], str) and parts[i + 1]:
                    section_content = parts[i + 1]
                    chunks = split_into_chunks(section_content)
 
                    # Create leaf nodes for each chunk and connect to the current section
                    for chunk in chunks:
                        leaf_node = len(node_list)
                        node_list.append({"title": "", "content": chunk, "type": "chunk"})
                        edges.append((section_node, leaf_node))

                        # Check for references in each chunk
                        refs = re.findall(r'\\ref\{(.*?)\}', chunk)
                        for ref in refs:
                            section_refs[ref].append(leaf_node)
        else:
            # Content outside of recognized sections
            continue

    return node_list, edges, section_refs



def extract_entities(text):
    doc = nlp(text)
    entities = [ent.text for ent in doc.ents]
    return entities

def compute_edges_with_entities(node_list):
    edge_list = []
    edge_attr_list = []
    node_entities = [extract_entities(node['content']) for node in node_list]

    for (i, entities_i), (j, entities_j) in itertools.combinations(enumerate(node_entities), 2):
        shared_entities = set(entities_i).intersection(set(entities_j)) 
        shared_count = len(shared_entities)
        if shared_count > 0:
            edge_list.append([i, j])
            edge_list.append([j, i])
            edge_attr_list.append(shared_count)
            edge_attr_list.append(shared_count)

    return edge_list, edge_attr_list

def create_pytorch_graph(node_list, initial_edges, section_refs):
    edge_list, edge_attr_list = compute_edges_with_entities(node_list)
    edge_list.extend(initial_edges)

    # Add cross-reference edges based on section_refs
    for ref, leaf_nodes in section_refs.items():
        for leaf_node in leaf_nodes:
            ref_index = next((i for i, n in enumerate(node_list) if n["title"] == ref), None)
            if ref_index is not None:
                edge_list.append((leaf_node, ref_index))
                edge_list.append((ref_index, leaf_node))

    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    num_nodes = len(node_list)
    x = torch.arange(num_nodes, dtype=torch.float32).unsqueeze(1)
    edge_attr = torch.tensor(edge_attr_list, dtype=torch.float32).unsqueeze(1)
    graph_data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

    return graph_data

# Example usage
if __name__ == "__main__":
    # Example LaTeX content with multiple nested sections
    tex_content = r"""
    \section{Introduction}
    This section provides an overview. As discussed in \ref{related_work}.
    \subsection{Background}
    Background content here.
    \subsubsection{Detailed Background}
    More specific background information.
    \subsection{Motivation}
    Motivation content. See also \ref{methods}.

    \section{Related Work}
    Related work discussed here.

    \section{Methods}
    Detailed methods are described here.
    """

    # Build graph
    node_list, initial_edges, section_refs = build_graph_from_tex_content(tex_content)
    graph = create_pytorch_graph(node_list, initial_edges, section_refs)

    # Output graph data
    print("Node List:", node_list)
    print(f"Edge Index (References):\n{graph.edge_index}")
    print(f"Edge Attributes (Shared Entities):\n{graph.edge_attr}")


Node List: [{'title': 'Sample Paper Title', 'content': 'Sample Paper Title', 'type': 'root'}, {'title': 'Introduction', 'content': 'Introduction', 'type': 'section_level_0'}, {'title': 'Background', 'content': 'Background', 'type': 'section_level_1'}, {'title': '', 'content': 'sub', 'type': 'chunk'}, {'title': 'Detailed Background', 'content': 'Detailed Background', 'type': 'section_level_2'}, {'title': '', 'content': 'sub', 'type': 'chunk'}, {'title': 'Motivation', 'content': 'Motivation', 'type': 'section_level_1'}, {'title': '', 'content': 'sub', 'type': 'chunk'}, {'title': 'Related Work', 'content': 'Related Work', 'type': 'section_level_0'}, {'title': 'Methods', 'content': 'Methods', 'type': 'section_level_0'}]
Edge Index (References):
tensor([[0, 1, 2, 2, 4, 1, 6, 0, 0],
        [1, 2, 3, 4, 5, 6, 7, 8, 9]])
Node Text Attributes:
['Sample Paper Title', 'Introduction', 'Background', 'sub', 'Detailed Background', 'sub', 'Motivation', 'sub', 'Related Work', 'Methods']


In [22]:
import re
import itertools
from collections import defaultdict
import torch
from torch_geometric.data import Data
import spacy

nlp = spacy.load("en_core_web_sm")

def split_into_chunks(text, max_tokens=100):
    words = text.split()
    chunks = [' '.join(words[i:i + max_tokens]) for i in range(0, len(words), max_tokens)]
    return chunks

def build_graph_from_tex_content(content, main_title="Main Title", chunk_size=100):
    node_list = []
    edges = []
    node_text = []  # Store text attributes for each node
    section_refs = defaultdict(list)
    section_stack = []  # Stack to manage the hierarchy of sections

    # Add the root node with the main title
    root_node = len(node_list)
    node_list.append({"title": main_title, "content": main_title, "type": "root"})
    node_text.append(main_title)  # Root node text attribute

    # Regex pattern to match any level of section (\section, \subsection, \subsubsection, etc.)
    pattern = r'(\\(sub)*section\{.*?\})'
    parts = re.split(pattern, content)

    current_parent_node = root_node  # Start with root as the initial parent
    for i in range(len(parts)):
        if isinstance(parts[i], str) and re.match(r'\\(sub)*section', parts[i]):
            # Calculate depth based on the number of "sub" prefixes
            depth = parts[i].count("sub")
            title_match = re.match(r'\\(sub)*section\{(.*?)\}', parts[i])
            if title_match:
                section_title = title_match.group(2)

                # Remove nodes from stack that are at the same or deeper level
                while section_stack and section_stack[-1][1] >= depth:
                    section_stack.pop()

                # Create a new intermediate node for this section with title as the text attribute
                section_node = len(node_list)
                node_list.append({"title": section_title, "content": section_title, "type": f"section_level_{depth}"})
                node_text.append(section_title)  # Add section title to node_text

                # Determine the parent node
                if section_stack:
                    parent_node = section_stack[-1][0]
                else:
                    parent_node = root_node  # Connect top-level sections to the root

                # Connect the current section to its parent
                edges.append((parent_node, section_node))

                # Push the current section onto the stack with its depth level
                section_stack.append((section_node, depth))

                # Determine if this is a final section without further subsections
                is_final_section = (
                    (i + 2 >= len(parts)) or  # No further sections
                    not re.match(r'\\(sub)*section', parts[i + 2]) or  # No more sections in parts
                    parts[i + 2].count("sub") <= depth  # Next section is a higher or same level
                )

                # If this is a final section, split its content into chunks as leaf nodes
                if is_final_section and i + 1 < len(parts) and isinstance(parts[i + 1], str) and parts[i + 1]:
                    section_content = parts[i + 1]
                    chunks = split_into_chunks(section_content, max_tokens=chunk_size)

                    # Create leaf nodes for each chunk and connect to the current section
                    for chunk in chunks:
                        leaf_node = len(node_list)
                        node_list.append({"title": "", "content": chunk, "type": "chunk"})
                        edges.append((section_node, leaf_node))
                        node_text.append(chunk)  # Add chunk content to node_text

                        # Check for references in each chunk
                        refs = re.findall(r'\\ref\{(.*?)\}', chunk)
                        for ref in refs:
                            section_refs[ref].append(leaf_node)
        else:
            # Content outside of recognized sections
            continue

    return node_list, edges, section_refs, node_text

def extract_entities(text):
    doc = nlp(text)
    entities = [ent.text for ent in doc.ents]
    return entities

def compute_edges_with_entities(node_list):
    edge_list = []
    edge_attr_list = []
    node_entities = [extract_entities(node['content']) for node in node_list]

    for (i, entities_i), (j, entities_j) in itertools.combinations(enumerate(node_entities), 2):
        shared_entities = set(entities_i).intersection(set(entities_j)) 
        shared_count = len(shared_entities)
        if shared_count > 0:
            edge_list.append([i, j])
            edge_list.append([j, i])
            edge_attr_list.append(shared_count)
            edge_attr_list.append(shared_count)

    return edge_list, edge_attr_list

def create_pytorch_graph(node_list, initial_edges, section_refs, node_text):
    edge_list, edge_attr_list = compute_edges_with_entities(node_list)
    edge_list.extend(initial_edges)

    # Add cross-reference edges based on section_refs
    for ref, leaf_nodes in section_refs.items():
        for leaf_node in leaf_nodes:
            ref_index = next((i for i, n in enumerate(node_list) if n["title"] == ref), None)
            if ref_index is not None:
                edge_list.append((leaf_node, ref_index))
                edge_list.append((ref_index, leaf_node))

    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    num_nodes = len(node_list)
    x = torch.arange(num_nodes, dtype=torch.float32).unsqueeze(1)  # Node feature matrix as indices

    graph_data = Data(x=x, edge_index=edge_index, node_text=node_text)  # Adding node_text as attribute

    return graph_data

# Example usage
if __name__ == "__main__":
    # Example LaTeX content with multiple nested sections
    tex_content = r"""
    \section{Introduction}
    This section provides an overview. As discussed in \ref{related_work}.
    \subsection{Background}
    Background content here.
    \subsubsection{Detailed Background}
    More specific background information.
    \subsection{Motivation}
    Motivation content. See also \ref{methods}.

    \section{Related Work}
    Related work discussed here.

    \section{Methods}
    Detailed methods are described here.
    """

    # Build graph
    node_list, initial_edges, section_refs, node_text = build_graph_from_tex_content(tex_content, main_title="Sample Paper Title", chunk_size=50)
    graph = create_pytorch_graph(node_list, initial_edges, section_refs, node_text)

    # Output graph data
    print("Node List:", node_list)
    print(f"Edge Index (References):\n{graph.edge_index}")
    print(f"Node Text Attributes:\n{graph.node_text}")


Node List: [{'title': 'Sample Paper Title', 'content': 'Sample Paper Title', 'type': 'root'}, {'title': 'Introduction', 'content': 'Introduction', 'type': 'section_level_0'}, {'title': 'Background', 'content': 'Background', 'type': 'section_level_1'}, {'title': '', 'content': 'sub', 'type': 'chunk'}, {'title': 'Detailed Background', 'content': 'Detailed Background', 'type': 'section_level_2'}, {'title': '', 'content': 'sub', 'type': 'chunk'}, {'title': 'Motivation', 'content': 'Motivation', 'type': 'section_level_1'}, {'title': '', 'content': 'sub', 'type': 'chunk'}, {'title': 'Related Work', 'content': 'Related Work', 'type': 'section_level_0'}, {'title': 'Methods', 'content': 'Methods', 'type': 'section_level_0'}]
Edge Index (References):
tensor([[0, 1, 2, 2, 4, 1, 6, 0, 0],
        [1, 2, 3, 4, 5, 6, 7, 8, 9]])
Node Text Attributes:
['Sample Paper Title', 'Introduction', 'Background', 'sub', 'Detailed Background', 'sub', 'Motivation', 'sub', 'Related Work', 'Methods']


In [32]:
import re

def parse_sections(content):
    # Pattern to match any section command, capturing the number of "sub" prefixes and section title
    pattern = r'(\\(sub)*section\{(.*?)\})'
    sections = []
    
    # Find all section commands and their titles
    matches = list(re.finditer(pattern, content))
    for i, match in enumerate(matches):
        section_command = match.group(1)  # Full section command
        title = match.group(3)  # Extract the section title
        start_index = match.start(1)  # Position in the content where the section starts
        end_index = matches[i + 1].start(1) if i + 1 < len(matches) else len(content)
        
        # Determine if this is a leaf node by checking if the next section has a higher or same level
        is_leaf = True  # Default to True (assume it's a leaf)
        if i + 1 < len(matches):
            next_command = matches[i + 1].group(1)
            # It's a leaf only if the next section is of a higher or equal hierarchy
            is_leaf = next_command.count("sub") <= section_command.count("sub")
        
        sections.append({
            "title": title,
            "start_index": start_index,
            "end_index": end_index,
            "is_leaf": is_leaf
        })
    
    return sections

# Example usage
if __name__ == "__main__":
    tex_content = r"""
    \section{Introduction}
    This section provides an overview. As discussed in \ref{related_work}.
    \subsection{Background}
    Background content here.
    \subsubsection{Detailed Background}
    More specific background information.
    \subsubsection{Preliminaries}
    More specific details
    \subsection{Motivation}
    Motivation content. See also \ref{methods}.

    \section{Related Work}
    Related work discussed here.

    \section{Methods}
    Detailed methods are described here.
    """

    sections = parse_sections(tex_content)
    for sec in sections:
        content_type = "Leaf" if sec["is_leaf"] else "Non-leaf"
        print(f"Title: {sec['title']}, Start Index: {sec['start_index']}, End Index: {sec['end_index']}, Type: {content_type}")


Title: Introduction, Start Index: 5, End Index: 107, Type: Non-leaf
Title: Background, Start Index: 107, End Index: 164, Type: Non-leaf
Title: Detailed Background, Start Index: 164, End Index: 246, Type: Leaf
Title: Preliminaries, Start Index: 246, End Index: 306, Type: Leaf
Title: Motivation, Start Index: 306, End Index: 383, Type: Leaf
Title: Related Work, Start Index: 383, End Index: 444, Type: Leaf
Title: Methods, Start Index: 444, End Index: 507, Type: Leaf


In [61]:
import re
from collections import defaultdict
import torch
from torch_geometric.data import Data

def split_into_chunks(text, max_tokens=50):
    words = text.split()
    chunks = [' '.join(words[i:i + max_tokens]) for i in range(0, len(words), max_tokens)]
    return chunks

def parse_sections(content):
    # Pattern to match any section command, capturing the number of "sub" prefixes and section title
    pattern = r'(\\(sub)*section\{(.*?)\})'
    sections = []
    
    # Find all section commands and their titles
    matches = list(re.finditer(pattern, content))
    for i, match in enumerate(matches):
        section_command = match.group(1)  # Full section command
        title = match.group(3)  # Extract the section title
        start_index = match.start(1)  # Position in the content where the section starts
        end_index = matches[i + 1].start(1) if i + 1 < len(matches) else len(content)
        
        # Determine if this is a leaf node by checking if the next section has a higher or same level
        is_leaf = True  # Default to True (assume it's a leaf)
        if i + 1 < len(matches):
            next_command = matches[i + 1].group(1)
            # It's a leaf only if the next section is of a higher or equal hierarchy
            is_leaf = next_command.count("sub") <= section_command.count("sub")
        
        sections.append({
            "title": title,
            "start_index": start_index,
            "end_index": end_index,
            "is_leaf": is_leaf,
            "depth": section_command.count("sub")  # Depth indicates level of subsection
        })
    
    return sections

def build_graph_from_sections(sections, content, main_title="Main Title", chunk_size=50):
    node_list = []
    edges = []
    node_text = []  # Store text attributes for each node
    section_refs = defaultdict(list)
    section_dict = {}  # Dictionary to map titles to node indices for referencing

    # Add the root node with the main title
    root_node = len(node_list)
    node_list.append({"title": main_title, "content": main_title, "type": "root"})
    node_text.append(main_title)  # Root node text attribute

    # Stack to maintain hierarchy of sections
    stack = [(root_node, -1)]  # (node_index, depth), with root node at depth -1

    for section in sections:
        # Create node for each section title
        section_node = len(node_list)
        node_list.append({"title": section["title"], "content": section["title"], "type": "section"})
        node_text.append(section["title"])  # Add section title to node_text
        section_dict[section["title"]] = section_node  # Map title to node index for cross-references

        # Pop items from the stack until we reach the correct parent level
        while stack and stack[-1][1] >= section["depth"]:
            stack.pop()

        # The top of the stack is the parent node for the current section
        parent_node = stack[-1][0]
        edges.append((parent_node, section_node))

        # Push the current section onto the stack
        stack.append((section_node, section["depth"]))

        # If the section is a leaf, split its content into chunks
        if section["is_leaf"]:
            section_content = content[section["start_index"]:section["end_index"]]
            chunks = split_into_chunks(section_content, max_tokens=chunk_size)

            # Create a node for each chunk and link it to the section
            for chunk in chunks:
                chunk_node = len(node_list)
                node_list.append({"title": "", "content": chunk, "type": "chunk"})
                node_text.append(chunk)
                edges.append((section_node, chunk_node))

                # Check for references in each chunk
                refs = re.findall(r'\\ref\{(.*?)\}', chunk)
                for ref in refs:
                    if ref in section_dict:
                        ref_node = section_dict[ref]
                        edges.append((chunk_node, ref_node))  # Add edge from chunk to referenced section

    return node_list, edges, node_text

def create_pytorch_graph(node_list, edges, node_text):
    # Convert edges to tensor format for PyTorch Geometric
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    num_nodes = len(node_list)
    x = torch.arange(num_nodes, dtype=torch.float32).unsqueeze(1)  # Node feature matrix as indices

    # Create the final PyTorch Geometric Data object
    graph_data = Data(x=x, edge_index=edge_index, node_text=node_text)  # Adding node_text as attribute

    return graph_data

# Example usage
if __name__ == "__main__":
    tex_content = r"""
    \section{Introduction}
    This section provides an overview. As discussed in \ref{related_work}.
    \subsection{Background}
    Background content here.
    \subsubsection{Detailed Background}
    More specific background information.
    \subsubsection{Preliminaries}
    More specific details
    \subsection{Motivation}
    Motivation content. See also \ref{methods}.

    \section{Related Work}
    Related work discussed here.

    \section{Methods}
    Detailed methods are described here.
    """

    # Step 1: Parse sections and identify leaf nodes
    sections = parse_sections(tex_content)

    # Step 2: Build the graph structure with nodes, edges, and node_text
    node_list, edges, node_text = build_graph_from_sections(sections, tex_content, main_title="Sample Paper Title", chunk_size=50)

    # Step 3: Create the PyTorch Geometric graph object
    graph = create_pytorch_graph(node_list, edges, node_text)

    # Output graph data for verification
    print("Node List:", node_list)
    print(f"Edge Index (References):\n{graph.edge_index}")
    print(f"Node Text Attributes:\n{graph.node_text}")
    print(graph)


Node List: [{'title': 'Sample Paper Title', 'content': 'Sample Paper Title', 'type': 'root'}, {'title': 'Introduction', 'content': 'Introduction', 'type': 'section'}, {'title': 'Background', 'content': 'Background', 'type': 'section'}, {'title': 'Detailed Background', 'content': 'Detailed Background', 'type': 'section'}, {'title': '', 'content': '\\subsubsection{Detailed Background} More specific background information.', 'type': 'chunk'}, {'title': 'Preliminaries', 'content': 'Preliminaries', 'type': 'section'}, {'title': '', 'content': '\\subsubsection{Preliminaries} More specific details', 'type': 'chunk'}, {'title': 'Motivation', 'content': 'Motivation', 'type': 'section'}, {'title': '', 'content': '\\subsection{Motivation} Motivation content. See also \\ref{methods}.', 'type': 'chunk'}, {'title': 'Related Work', 'content': 'Related Work', 'type': 'section'}, {'title': '', 'content': '\\section{Related Work} Related work discussed here.', 'type': 'chunk'}, {'title': 'Methods', 'con

In [62]:
import re

def get_section_labels(content):
    # Pattern to match any section command and an optional label
    pattern = r'(\\(sub)*section\{(.*?)\})(?:\s*\\label\{(.*?)\})?'
    section_labels = {}

    # Find all section commands with optional labels
    matches = re.finditer(pattern, content)
    for match in matches:
        section_title = match.group(3)  # Extract the section title
        section_label = match.group(4)  # Extract the optional label, if present
        if section_label:
            section_labels[section_title] = section_label

    return section_labels

# Example usage
if __name__ == "__main__":
    tex_content = r"""
    \section{Introduction}
    This section provides an overview. As discussed in \ref{related_work}.
    \subsection{Background}
    Background content here.
    \subsubsection{Detailed Background}
    More specific background information.
    \subsubsection{Preliminaries}
    More specific details 
    \subsection{Motivation}
    Motivation content. See also \ref{methods}.

    \section{Related Work}\label{related_work}
    Related work discussed here.

    \section{Methods}\label{methods}
    Detailed methods are described here.
    """

    section_labels = get_section_labels(tex_content)
    print("Section Labels Dictionary:", section_labels)


Section Labels Dictionary: {'Related Work': 'related_work', 'Methods': 'methods'}


In [66]:
import re
import torch
from typing import List, Dict, Tuple

def build_reference_edges(node_texts: List[str], edge_index: torch.Tensor, section_labels: Dict[str, str]) -> torch.Tensor:
    # Create reverse mapping from labels to node indices
    label_to_node = {}
    
    # First pass: build mapping of labels to node indices
    for idx, text in enumerate(node_texts):
        # Pattern to match section command with label
        section_pattern = r'\\(?:sub)*section\{(.*?)\}(?:\s*\\label\{(.*?)\})?'
        matches = re.search(section_pattern, text)
        if matches:
            section_title = matches.group(1)
            if section_title in section_labels:
                label = section_labels[section_title]
                label_to_node[label] = idx

    # Convert existing edges to list of tuples
    existing_edges = list(zip(edge_index[0].tolist(), edge_index[1].tolist()))
    all_edges = existing_edges.copy()
    
    # Second pass: find \ref{} commands and add edges
    ref_pattern = r'\\ref\{(.*?)\}'
    for source_idx, text in enumerate(node_texts):
        refs = re.finditer(ref_pattern, text)
        for ref in refs:
            label = ref.group(1)
            if label in label_to_node:
                target_idx = label_to_node[label]
                new_edge = (source_idx, target_idx)
                if new_edge not in all_edges:
                    all_edges.append(new_edge)

    # Convert back to PyTorch tensor
    if all_edges:
        source_nodes = [edge[0] for edge in all_edges]
        target_nodes = [edge[1] for edge in all_edges]
        new_edge_index = torch.tensor([source_nodes, target_nodes], dtype=torch.long)
    else:
        new_edge_index = edge_index.clone()
    
    return new_edge_index

# Example usage:
if __name__ == "__main__":
    # Update graph edge_index
    graph.edge_index = build_reference_edges(graph.node_text, graph.edge_index, section_labels)
    print("Updated edges:", graph.edge_index)

Updated edges: tensor([[ 0,  1,  2,  3,  2,  5,  1,  7,  0,  9,  0, 11,  8],
        [ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 12]])


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from torch_geometric.utils import to_networkx

def visualize_pyg_graph_with_directed_edges(data):
    # Convert the PyTorch Geometric Data object to a NetworkX directed graph
    G = to_networkx(data, to_undirected=False)  # Keep it directed

    # Assign labels to each node from node_text attribute
    labels = {i: data.node_text[i] for i in range(len(data.node_text))}

    # Draw the directed graph with arrows
    pos = nx.spring_layout(G)  # Layout for visualization
    plt.figure(figsize=(12, 8))
    nx.draw(G, pos, with_labels=False, node_size=3000, node_color="skyblue", font_size=10, font_weight="bold", arrows=True)
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=9)
    
    # Add title and show the plot
    plt.title("Directed Graph Visualization with Node Text Attributes")
    plt.show()

visualize_pyg_graph_with_directed_edges(graph)


In [8]:
import torch 
from sentence_transformers import SentenceTransformer
from transformers import (
    T5Model, T5Tokenizer, 
    AutoModel, AutoTokenizer,
    PreTrainedModel, PreTrainedTokenizer
)
from typing import List, Union, Dict, Any
import numpy as np
from FlagEmbedding import BGEM3FlagModel

class BaseRetriever:
    def __init__(self, device: str = 'cuda' if torch.cuda.is_available() else 'cpu'):
        self.device = device
        self.model = None
        self.tokenizer = None
    
    def encode(self, texts: List[str], batch_size: int = 32, **kwargs) -> np.ndarray:
        raise NotImplementedError

    def compute_similarity(self, query_embeddings: np.ndarray, doc_embeddings: np.ndarray) -> np.ndarray:
        """Compute cosine similarity between query and document embeddings"""
        return np.dot(query_embeddings, doc_embeddings.T)

    def search(self, query: str, documents: List[str], top_k: int = 5) -> List[Dict[str, Any]]:
        """Basic search functionality"""
        query_embedding = self.encode([query])
        doc_embeddings = self.encode(documents)
        
        scores = self.compute_similarity(query_embedding, doc_embeddings)[0]
        top_idx = np.argsort(scores)[::-1][:top_k]
        
        results = []
        for idx in top_idx:
            results.append({
                'document': documents[idx],
                'score': float(scores[idx]),
                'index': int(idx)
            })
        return results

class MiniLMRetriever(BaseRetriever):
    def __init__(self, device: str = 'cuda' if torch.cuda.is_available() else 'cpu'):
        super().__init__(device)
        self.model = SentenceTransformer('sentence-transformers/all-MiniLM-L12-v2').to(device)
    
    def encode(self, texts: List[str], batch_size: int = 32, **kwargs) -> np.ndarray:
        return self.model.encode(texts, batch_size=batch_size, device=self.device, **kwargs)

class SentenceBERTRetriever(BaseRetriever):
    def __init__(self, device: str = 'cuda' if torch.cuda.is_available() else 'cpu'):
        super().__init__(device)
        self.model = SentenceTransformer('sentence-transformers/bert-base-nli-mean-tokens').to(device)
    
    def encode(self, texts: List[str], batch_size: int = 32, **kwargs) -> np.ndarray:
        return self.model.encode(texts, batch_size=batch_size, device=self.device, **kwargs)

class LaBSERetriever(BaseRetriever):
    def __init__(self, device: str = 'cuda' if torch.cuda.is_available() else 'cpu'):
        super().__init__(device)
        self.model = SentenceTransformer('sentence-transformers/LaBSE').to(device)
    
    def encode(self, texts: List[str], batch_size: int = 32, **kwargs) -> np.ndarray:
        return self.model.encode(texts, batch_size=batch_size, device=self.device, **kwargs)

class mContrieverRetriever(BaseRetriever):
    def __init__(self, device: str = 'cuda' if torch.cuda.is_available() else 'cpu'):
        super().__init__(device)
        self.model = AutoModel.from_pretrained('facebook/mcontriever-msmarco').to(device)
        self.tokenizer = AutoTokenizer.from_pretrained('facebook/mcontriever-msmarco')
        
    def mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    
    def encode(self, texts: List[str], batch_size: int = 32, **kwargs) -> np.ndarray:
        all_embeddings = []
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            encoded_input = self.tokenizer(batch_texts, padding=True, truncation=True, 
                                        max_length=512, return_tensors='pt').to(self.device)
            with torch.no_grad():
                model_output = self.model(**encoded_input)
            sentence_embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])
            all_embeddings.append(sentence_embeddings.cpu().numpy())
        return np.vstack(all_embeddings)

class T5Retriever(BaseRetriever):
    def __init__(self, device: str = 'cuda' if torch.cuda.is_available() else 'cpu'):
        super().__init__(device)
        self.model = T5Model.from_pretrained('t5-base').to(device)
        self.tokenizer = T5Tokenizer.from_pretrained('t5-base')
        
    def encode(self, texts: List[str], batch_size: int = 32, **kwargs) -> np.ndarray:
        all_embeddings = []
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            encoded_input = self.tokenizer(batch_texts, padding=True, truncation=True, 
                                        max_length=512, return_tensors='pt').to(self.device)
            with torch.no_grad():
                outputs = self.model.encoder(**encoded_input)
                # Use [CLS] token embedding or mean pooling
                embeddings = outputs.last_hidden_state.mean(dim=1)
            all_embeddings.append(embeddings.cpu().numpy())
        return np.vstack(all_embeddings)

class E5Retriever(BaseRetriever):
    def __init__(self, device: str = 'cuda' if torch.cuda.is_available() else 'cpu'):
        super().__init__(device)
        self.model = AutoModel.from_pretrained('intfloat/e5-base').to(device)
        self.tokenizer = AutoTokenizer.from_pretrained('intfloat/e5-base')
    
    def encode(self, texts: List[str], batch_size: int = 32, **kwargs) -> np.ndarray:
        all_embeddings = []
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            # Add prefix for queries: "query: "
            encoded_input = self.tokenizer(batch_texts, padding=True, truncation=True,
                                        max_length=512, return_tensors='pt').to(self.device)
            with torch.no_grad():
                outputs = self.model(**encoded_input)
                embeddings = outputs.last_hidden_state[:, 0]  # Use [CLS] token
            all_embeddings.append(embeddings.cpu().numpy())
        return np.vstack(all_embeddings)

class BGEM3FlagRetriever(BaseRetriever):
    def __init__(self, device: str = 'cuda' if torch.cuda.is_available() else 'cpu'):
        super().__init__(device)
        self.model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
    
    def encode(self, texts: List[str], batch_size: int = 32, **kwargs) -> np.ndarray:
        outputs = self.model.encode(
            texts, batch_size=batch_size, max_length=8192, **kwargs
        )
        return outputs['dense_vecs']
    
class SPARRetriever(BaseRetriever):
    def __init__(self, model_name="facebook/spar-paq-bm25-lexmodel-context-encoder", device: str = 'cuda' if torch.cuda.is_available() else 'cpu'):
        super().__init__(device)
        self.model = AutoModel.from_pretrained(model_name).to(device)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    def encode(self, texts: List[str], batch_size=32, **kwargs) -> np.ndarray:
        embeddings = []
        with torch.no_grad():
            for i in range(0, len(texts), batch_size):
                batch = texts[i:i + batch_size]
                inputs = self.tokenizer(batch, padding=True, truncation=True, 
                                        max_length=512, return_tensors="pt").to(self.device)
                outputs = self.model(**inputs)
                cls_embeddings = outputs.last_hidden_state[:, 0, :]  # Use CLS token
                embeddings.append(cls_embeddings.cpu().numpy())
        return np.vstack(embeddings)




# Example usage
def test_retrievers():
    # Sample documents
    documents = [
        "The quick brown fox jumps over the lazy dog.",
        "Machine learning is a subset of artificial intelligence.",
        "Python is a popular programming language.",
        "Neural networks are inspired by biological neurons.",
        "Deep learning has revolutionized computer vision."
    ]
    
    query = "What is artificial intelligence?"
    
    # Initialize retrievers
    retrievers = {
        "MiniLM": MiniLMRetriever(),
        "SentenceBERT": SentenceBERTRetriever(),
        "LaBSE": LaBSERetriever(),
        "mContriever": mContrieverRetriever(),
        "T5": T5Retriever(),
        "E5": E5Retriever(),
        "BGE-M3-Flag": BGEM3FlagRetriever(),
        "SPAR": SPARRetriever()
    }
    
    # Test each retriever
    for name, retriever in retrievers.items():
        print(f"\nResults for {name}:")
        results = retriever.search(query, documents, top_k=3)
        for rank, result in enumerate(results, 1):
            print(f"{rank}. Score: {result['score']:.4f} - {result['document']}")

if __name__ == "__main__":
    test_retrievers()


Some weights of BertModel were not initialized from the model checkpoint at facebook/mcontriever-msmarco and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]


Results for MiniLM:
1. Score: 0.5969 - Machine learning is a subset of artificial intelligence.
2. Score: 0.3792 - Neural networks are inspired by biological neurons.
3. Score: 0.2676 - Python is a popular programming language.

Results for SentenceBERT:
1. Score: 206.2514 - Machine learning is a subset of artificial intelligence.
2. Score: 148.1823 - Neural networks are inspired by biological neurons.
3. Score: 128.4384 - Deep learning has revolutionized computer vision.

Results for LaBSE:
1. Score: 0.5219 - Machine learning is a subset of artificial intelligence.
2. Score: 0.2653 - Deep learning has revolutionized computer vision.
3. Score: 0.2266 - Neural networks are inspired by biological neurons.

Results for mContriever:
1. Score: 2.2695 - Machine learning is a subset of artificial intelligence.
2. Score: 1.6350 - Deep learning has revolutionized computer vision.
3. Score: 1.2016 - Neural networks are inspired by biological neurons.

Results for T5:
1. Score: 7.5670 - Machine 

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


1. Score: 0.6606 - Machine learning is a subset of artificial intelligence.
2. Score: 0.5234 - Deep learning has revolutionized computer vision.
3. Score: 0.4727 - Python is a popular programming language.

Results for SPAR:
1. Score: 4721.9404 - Deep learning has revolutionized computer vision.
2. Score: 4719.2690 - The quick brown fox jumps over the lazy dog.
3. Score: 4717.1836 - Machine learning is a subset of artificial intelligence.


In [ ]:
import torch
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModel
from typing import List, Dict, Any, Tuple
import scipy.sparse as sp

class HybridRetriever:
    def __init__(self, 
                 lambda_weight: float = 0.8,
                 specter2_model: str = "allenai/specter2_base",
                 device: str = "cuda" if torch.cuda.is_available() else "cpu"):
        """
        Initialize hybrid retriever combining sparse BOW and dense SPECTER2 embeddings.
        Args:
            lambda_weight: Weight for dense embeddings (1-lambda for sparse)
            specter2_model: HuggingFace model name for SPECTER2
            device: Device to run model on
        """
        self.lambda_weight = lambda_weight
        self.device = device

        # Initialize sparse components
        self.tfidf = TfidfVectorizer(stop_words='english')
        
        # Initialize dense components
        self.tokenizer = AutoTokenizer.from_pretrained(specter2_model)
        self.model = AutoModel.from_pretrained(specter2_model).to(device)
        
        # Storage for document embeddings
        self.doc_sparse_embeddings = None
        self.doc_dense_embeddings = None
        
    def _get_dense_embeddings(self, texts: List[str]) -> np.ndarray:
        """Get dense embeddings using SPECTER2"""
        embeddings = []
        
        with torch.no_grad():
            for text in texts:
                # Tokenize and move to device
                inputs = self.tokenizer(text, 
                                      padding=True, 
                                      truncation=True,
                                      max_length=512,
                                      return_tensors="pt").to(self.device)
                
                # Get model output
                outputs = self.model(**inputs)
                
                # Use CLS token embedding
                embeddings.append(outputs.last_hidden_state[:, 0, :].cpu().numpy())
        
        return np.vstack(embeddings)
    
    def _get_sparse_embeddings(self, texts: List[str], fit: bool = False) -> sp.csr_matrix:
        """Get sparse TF-IDF embeddings"""
        if fit:
            return self.tfidf.fit_transform(texts)
        return self.tfidf.transform(texts)
    
    def index_documents(self, documents: List[str]):
        """Index documents to build sparse and dense representations"""
        # Get sparse embeddings
        self.doc_sparse_embeddings = self._get_sparse_embeddings(documents, fit=True)
        
        # Get dense embeddings
        self.doc_dense_embeddings = self._get_dense_embeddings(documents)
        
    def _cosine_similarity(self, a: np.ndarray, b: np.ndarray) -> np.ndarray:
        """Compute cosine similarity between embeddings"""
        norm_a = np.linalg.norm(a, axis=1)
        norm_b = np.linalg.norm(b, axis=1)
        return np.dot(a, b.T) / np.outer(norm_a, norm_b)

    def _sparse_cosine_similarity(self, a: sp.csr_matrix, b: sp.csr_matrix) -> np.ndarray:
        """Compute cosine similarity between sparse matrices"""
        norm_a = np.sqrt(a.multiply(a).sum(axis=1))
        norm_b = np.sqrt(b.multiply(b).sum(axis=1))
        return np.array(a.dot(b.T).todense()) / np.outer(norm_a, norm_b)

    def search(self, 
              query: str, 
              top_k: int = 10) -> List[Dict[str, Any]]:
        """
        Search for documents relevant to query using hybrid retrieval
        Args:
            query: Query text
            top_k: Number of results to return
        Returns:
            List of dicts with document indices and scores
        """
        # Get query embeddings
        query_sparse = self._get_sparse_embeddings([query])
        query_dense = self._get_dense_embeddings([query])
        
        # Calculate similarities
        sparse_scores = self._sparse_cosine_similarity(
            query_sparse, 
            self.doc_sparse_embeddings
        )[0]
        
        dense_scores = self._cosine_similarity(
            query_dense, 
            self.doc_dense_embeddings
        )[0]
        
        # Combine scores using weighted sum
        combined_scores = (
            self.lambda_weight * dense_scores + 
            (1 - self.lambda_weight) * sparse_scores
        )
        
        # Get top-k indices and scores
        top_indices = np.argsort(combined_scores)[::-1][:top_k]
        top_scores = combined_scores[top_indices]
        
        results = []
        for idx, score in zip(top_indices, top_scores):
            results.append({
                "index": int(idx),
                "score": float(score),
                "sparse_score": float(sparse_scores[idx]),
                "dense_score": float(dense_scores[idx])
            })
            
        return results

# Example usage
def test_hybrid_retriever():
    # Sample documents
    documents = [
        "The effects of cystic fibrosis on lung function",
        "Treatment options for respiratory infections",
        "Genetic factors in cystic fibrosis",
        "Impact of diet on cystic fibrosis patients",
        "Latest research in cystic fibrosis treatments"
    ]
    
    # Initialize retriever with 0.8 weight for dense embeddings
    retriever = HybridRetriever(lambda_weight=0.8)
    
    # Index documents
    retriever.index_documents(documents)
    
    # Test search
    query = "What are the genetic causes of cystic fibrosis?"
    results = retriever.search(query, top_k=3)
    
    print(f"\nQuery: {query}")
    for rank, result in enumerate(results, 1):
        print(f"\nRank {rank}:")
        print(f"Document: {documents[result['index']]}")
        print(f"Combined Score: {result['score']:.4f}")
        print(f"Dense Score: {result['dense_score']:.4f}")
        print(f"Sparse Score: {result['sparse_score']:.4f}")

if __name__ == "__main__":
    test_hybrid_retriever()

In [25]:
from collections import Counter
import math
import numpy as np
from FlagEmbedding import BGEM3FlagModel
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration, AutoTokenizer, AutoModel
from typing import List, Tuple, Dict, Any
import scipy.sparse as sp


class BaseRetriever:
    def _tokenize(self, text: str) -> List[str]:
        """Simple tokenization by splitting on whitespace and converting to lowercase"""
        return text.lower().split()

    def score(self, query: str) -> List[Tuple[int, float]]:
        raise NotImplementedError

class BM25(BaseRetriever):
    def __init__(self, chunks: List[str], k1: float = 1.5, b: float = 0.75):
        """
        Initialize BM25 with document chunks and parameters
        """
        self.k1 = k1
        self.b = b
        self.chunks = chunks
        self.N = len(chunks)
        
        # Calculate document frequencies and lengths
        self.doc_freqs = {}
        self.doc_lengths = []
        total_len = 0
        
        for chunk in chunks:
            terms = self._tokenize(chunk)
            length = len(terms)
            self.doc_lengths.append(length)
            total_len += length
            
            # Update document frequencies
            for term in set(terms):
                self.doc_freqs[term] = self.doc_freqs.get(term, 0) + 1
        
        self.avgdl = total_len / self.N if self.N > 0 else 0

    def score(self, query: str) -> List[Tuple[int, float]]:
        query_terms = self._tokenize(query)
        scores = []
        
        for idx, chunk in enumerate(self.chunks):
            score = 0
            chunk_terms = self._tokenize(chunk)
            term_freqs = Counter(chunk_terms)
            doc_len = self.doc_lengths[idx]
            
            for term in query_terms:
                if term not in self.doc_freqs:
                    continue
                
                # Calculate IDF
                idf = math.log((self.N - self.doc_freqs[term] + 0.5) / 
                             (self.doc_freqs[term] + 0.5) + 1)
                
                # Calculate normalized term frequency
                tf = term_freqs[term]
                norm_tf = ((tf * (self.k1 + 1)) / 
                          (tf + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)))
                
                score += idf * norm_tf
            
            scores.append((idx, score))
        
        return sorted(scores, key=lambda x: x[1], reverse=True)

class TFIDF(BaseRetriever):
    def __init__(self, chunks: List[str]):
        """Initialize TF-IDF with document chunks"""
        self.chunks = chunks
        self.N = len(chunks)
        
        # Calculate document frequencies
        self.doc_freqs = {}
        for chunk in chunks:
            terms = set(self._tokenize(chunk))
            for term in terms:
                self.doc_freqs[term] = self.doc_freqs.get(term, 0) + 1

    def score(self, query: str) -> List[Tuple[int, float]]:
        query_terms = self._tokenize(query)
        scores = []
        
        for idx, chunk in enumerate(self.chunks):
            score = 0
            chunk_terms = self._tokenize(chunk)
            term_freqs = Counter(chunk_terms)
            
            for term in query_terms:
                if term not in self.doc_freqs:
                    continue
                
                # Calculate TF-IDF
                tf = term_freqs[term]
                idf = math.log(self.N / self.doc_freqs[term] + 1)
                score += tf * idf
            
            scores.append((idx, score))
        
        return sorted(scores, key=lambda x: x[1], reverse=True)

class BooleanRetriever(BaseRetriever):
    def __init__(self, chunks: List[str]):
        """Initialize Boolean Retriever with document chunks"""
        self.chunks = chunks
        self.chunk_terms = [set(self._tokenize(chunk)) for chunk in chunks]

    def score(self, query: str) -> List[Tuple[int, float]]:
        query_terms = set(self._tokenize(query))
        scores = []
        
        for idx, chunk_terms in enumerate(self.chunk_terms):
            # Coordination factor: ratio of matching terms
            matching_terms = len(query_terms & chunk_terms)
            score = matching_terms / len(query_terms) if query_terms else 0
            scores.append((idx, score))
        
        return sorted(scores, key=lambda x: x[1], reverse=True)

class ExtendedBoolean(BaseRetriever):
    def __init__(self, chunks: List[str], p: float = 2.0):
        """
        Initialize Extended Boolean Retriever with document chunks
        p: p-norm parameter (typically 2.0 for Euclidean norm)
        """
        self.chunks = chunks
        self.p = p
        self.N = len(chunks)
        
        # Calculate IDF weights for term importance
        self.doc_freqs = {}
        for chunk in chunks:
            terms = set(self._tokenize(chunk))
            for term in terms:
                self.doc_freqs[term] = self.doc_freqs.get(term, 0) + 1

    def score(self, query: str) -> List[Tuple[int, float]]:
        query_terms = self._tokenize(query)
        scores = []
        
        for idx, chunk in enumerate(self.chunks):
            chunk_terms = self._tokenize(chunk)
            term_freqs = Counter(chunk_terms)
            
            # Calculate weighted p-norm similarity
            sum_weights = 0
            for term in query_terms:
                if term not in self.doc_freqs:
                    continue
                
                # Calculate term weight using TF-IDF
                tf = term_freqs[term]
                idf = math.log(self.N / self.doc_freqs[term] + 1)
                weight = tf * idf
                sum_weights += weight ** self.p
            
            # Final score using p-norm
            score = (sum_weights / len(query_terms)) ** (1/self.p) if query_terms else 0
            scores.append((idx, score))
        
        return sorted(scores, key=lambda x: x[1], reverse=True)

class BGEM3Retriever(BaseRetriever):
    def __init__(self, chunks: List[str], model_name: str = 'BAAI/bge-m3', use_fp16: bool = True):
        """
        Initialize BGE-M3 retriever with document chunks and model.
        """
        self.chunks = chunks
        self.model = BGEM3FlagModel(model_name, use_fp16=use_fp16)
        
        # Precompute lexical weights for chunks
        self.chunk_lexical_weights = [
            self.model.encode([chunk], return_dense=False, return_sparse=True)['lexical_weights'][0]
            for chunk in chunks
        ]

    def score(self, query: str) -> List[Tuple[int, float]]:
        query_lexical_weights = self.model.encode([query], return_dense=False, return_sparse=True)['lexical_weights'][0]
        scores = []

        for idx, chunk_weights in enumerate(self.chunk_lexical_weights):
            # Use lexical matching score from BGE-M3
            score = self.model.compute_lexical_matching_score(query_lexical_weights, chunk_weights)
            scores.append((idx, score))
        
        return sorted(scores, key=lambda x: x[1], reverse=True)

class Doc2QueryRetriever(BaseRetriever):
    def __init__(self, 
                 chunks: List[str], 
                 model_name: str = "doc2query/msmarco-t5-base-v1",
                 device: str = 'cuda' if torch.cuda.is_available() else 'cpu',
                 num_queries: int = 3):
        """
        Initialize the Doc2Query retriever.
        """
        self.chunks = chunks
        self.device = device
        self.num_queries = num_queries
        self.tokenizer = T5Tokenizer.from_pretrained(model_name)
        self.model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)
        
        # Expand documents with synthetic queries
        print("Generating synthetic queries for all document chunks...")
        self.expanded_chunks = [self._expand_document(chunk) for chunk in chunks]

    def _generate_queries(self, document: str) -> List[str]:
        inputs = self.tokenizer.encode(
            document,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model.generate(
                inputs, 
                max_length=64,  # Max length for synthetic queries
                num_return_sequences=self.num_queries,
                do_sample=True,  # Sampling for diverse queries
                top_k=50,
                top_p=0.95
            )
        
        queries = [self.tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
        return queries

    def _expand_document(self, document: str) -> str:
        synthetic_queries = self._generate_queries(document)
        expanded_document = document + " " + " ".join(synthetic_queries)
        return expanded_document

    def score(self, query: str) -> List[Tuple[int, float]]:
        retriever = BM25(self.expanded_chunks)  # Use BM25 with expanded chunks
        return retriever.score(query)

class SPLADERetriever(BaseRetriever):
    """SPLADE++ Sparse Retriever."""

    def __init__(self, 
                 model_name: str = "naver/splade-cocondenser-ensembledistil", 
                 device: str = "cuda" if torch.cuda.is_available() else "cpu"):
        super().__init__()
        self.device = device

        # Initialize SPLADE tokenizer and model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(device)

        # Storage for sparse document representations
        self.doc_sparse_embeddings = None

    def _get_sparse_embeddings(self, texts: List[str]) -> sp.csr_matrix:
        """Generate sparse embeddings for a list of texts."""
        sparse_embeddings = []

        self.model.eval()
        with torch.no_grad():
            for text in texts:
                # Tokenize input
                inputs = self.tokenizer(
                    text,
                    padding=True,
                    truncation=True,
                    max_length=512,
                    return_tensors="pt"
                ).to(self.device)

                # Generate sparse logits
                outputs = self.model(**inputs)
                logits = outputs.last_hidden_state.sum(dim=1)  # Sum over token dimension

                # Convert logits to sparse representation
                sparse_vector = logits.squeeze(0).cpu().numpy()
                sparse_embeddings.append(sparse_vector)

        # Convert to CSR sparse matrix
        return sp.csr_matrix(sparse_embeddings)

    def index_documents(self, documents: List[str]):
        """Index documents to create their sparse embeddings."""
        self.documents = documents
        self.doc_sparse_embeddings = self._get_sparse_embeddings(documents)
        print(f"Indexed {len(documents)} documents with SPLADE++.")

    def _sparse_cosine_similarity(self, query: sp.csr_matrix, documents: sp.csr_matrix) -> np.ndarray:
        """Compute cosine similarity between query and document sparse embeddings."""
        norm_query = np.sqrt(query.multiply(query).sum(axis=1))
        norm_docs = np.sqrt(documents.multiply(documents).sum(axis=1))
        similarity = np.array(query.dot(documents.T).todense()) / np.outer(norm_query, norm_docs)
        return similarity.squeeze()

    def search(self, query: str, top_k: int = 10) -> List[Dict[str, Any]]:
        """Retrieve top-k documents for the given query."""
        # Get sparse embedding for the query
        query_sparse = self._get_sparse_embeddings([query])

        # Calculate similarities
        scores = self._sparse_cosine_similarity(query_sparse, self.doc_sparse_embeddings)

        # Get top-k results
        top_indices = np.argsort(scores)[::-1][:top_k]
        top_scores = scores[top_indices]

        return [
            {"index": int(idx), "score": float(score), "document": self.documents[idx]}
            for idx, score in zip(top_indices, top_scores)
        ]
    
# Example usage
def test_retrievers():
    # Sample documents
    chunks = [
        "the quick brown fox jumps over the lazy dog",
        "a quick brown cat sleeps on the windowsill",
        "the lazy dog barks at the mailman",
        "a fox and a dog play in the garden"
    ]
    
    # Initialize retrievers
    retrievers = {
        "BM25": BM25(chunks),
        "TF-IDF": TFIDF(chunks),
        "Boolean": BooleanRetriever(chunks),
        "Extended Boolean": ExtendedBoolean(chunks),
        "BGE-M3 Sparse Retriever": BGEM3Retriever(chunks),
        "Doc2Query": Doc2QueryRetriever(chunks),
        "SPLADE++": SPLADERetriever()
    }
    
    # Index documents for retrievers that require indexing
    for name, retriever in retrievers.items():
        if hasattr(retriever, "index_documents"):
            print(f"\nIndexing documents for {name}...")
            retriever.index_documents(chunks)

    # Test queries
    queries = [
        "quick brown",
        "lazy dog",
        "fox garden"
    ]
    
    print("Document chunks:")
    for idx, chunk in enumerate(chunks):
        print(f"{idx}: {chunk}")
    
    for query in queries:
        print(f"\nQuery: '{query}'")
        for name, retriever in retrievers.items():
            print(f"\n{name} scores:")
            if hasattr(retriever, "search"):
                # Use the `search` method for retrievers that support it
                results = retriever.search(query, top_k=3)
                for rank, result in enumerate(results, 1):
                    print(f"{rank}. Score: {result['score']:.4f} - Document: {result['document']}")
            else:
                # Use the `score` method for retrievers without `search`
                scores = retriever.score(query)
                for rank, (chunk_idx, score) in enumerate(scores[:3], 1):  # Top 3
                    print(f"{rank}. Score: {score:.4f} - Document: {chunks[chunk_idx]}")


if __name__ == "__main__":
    test_retrievers()

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Generating synthetic queries for all document chunks...


Some weights of BertModel were not initialized from the model checkpoint at naver/splade-cocondenser-ensembledistil and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Indexing documents for SPLADE++...
Indexed 4 documents with SPLADE++.
Document chunks:
0: the quick brown fox jumps over the lazy dog
1: a quick brown cat sleeps on the windowsill
2: the lazy dog barks at the mailman
3: a fox and a dog play in the garden

Query: 'quick brown'

BM25 scores:
1. Score: 1.4055 - Document: a quick brown cat sleeps on the windowsill
2. Score: 1.3318 - Document: the quick brown fox jumps over the lazy dog
3. Score: 0.0000 - Document: the lazy dog barks at the mailman

TF-IDF scores:
1. Score: 2.1972 - Document: the quick brown fox jumps over the lazy dog
2. Score: 2.1972 - Document: a quick brown cat sleeps on the windowsill
3. Score: 0.0000 - Document: the lazy dog barks at the mailman

Boolean scores:
1. Score: 1.0000 - Document: the quick brown fox jumps over the lazy dog
2. Score: 1.0000 - Document: a quick brown cat sleeps on the windowsill
3. Score: 0.0000 - Document: the lazy dog barks at the mailman

Extended Boolean scores:
1. Score: 1.0986 - Docume

In [26]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
import scipy.sparse as sp
from typing import List, Dict, Any
from sklearn.feature_extraction.text import TfidfVectorizer
from src.model.CLEAR import ResidualDenseRetriever


class ScoreFusionRetriever:
    # Method in https://arxiv.org/pdf/2010.01195
    def __init__(self, 
                 lambda_weight: float = 0.8,
                 specter2_model: str = "allenai/specter2_base",
                 device: str = "cuda" if torch.cuda.is_available() else "cpu"):
        """
        Initialize hybrid retriever combining sparse BOW and dense SPECTER2 embeddings.
        """
        self.lambda_weight = lambda_weight
        self.device = device

        # Initialize sparse components
        self.tfidf = TfidfVectorizer(stop_words='english')
        
        # Initialize dense components
        self.tokenizer = AutoTokenizer.from_pretrained(specter2_model)
        self.model = AutoModel.from_pretrained(specter2_model).to(device)
        
        # Storage for document embeddings
        self.doc_sparse_embeddings = None
        self.doc_dense_embeddings = None
        
    def _get_dense_embeddings(self, texts: List[str]) -> np.ndarray:
        """Get dense embeddings using SPECTER2"""
        embeddings = []
        
        with torch.no_grad():
            for text in texts:
                # Tokenize and move to device
                inputs = self.tokenizer(text, 
                                      padding=True, 
                                      truncation=True,
                                      max_length=512,
                                      return_tensors="pt").to(self.device)
                
                # Get model output
                outputs = self.model(**inputs)
                
                # Use CLS token embedding
                embeddings.append(outputs.last_hidden_state[:, 0, :].cpu().numpy())
        
        return np.vstack(embeddings)
    
    def _get_sparse_embeddings(self, texts: List[str], fit: bool = False) -> sp.csr_matrix:
        """Get sparse TF-IDF embeddings"""
        if fit:
            return self.tfidf.fit_transform(texts)
        return self.tfidf.transform(texts)
    
    def index_documents(self, documents: List[str]):
        """Index documents to build sparse and dense representations"""
        # Get sparse embeddings
        self.doc_sparse_embeddings = self._get_sparse_embeddings(documents, fit=True)
        
        # Get dense embeddings
        self.doc_dense_embeddings = self._get_dense_embeddings(documents)
        
    def _cosine_similarity(self, a: np.ndarray, b: np.ndarray) -> np.ndarray:
        """Compute cosine similarity between embeddings"""
        norm_a = np.linalg.norm(a, axis=1)
        norm_b = np.linalg.norm(b, axis=1)
        return np.dot(a, b.T) / np.outer(norm_a, norm_b)

    def _sparse_cosine_similarity(self, a: sp.csr_matrix, b: sp.csr_matrix) -> np.ndarray:
        """Compute cosine similarity between sparse matrices"""
        norm_a = np.sqrt(a.multiply(a).sum(axis=1))
        norm_b = np.sqrt(b.multiply(b).sum(axis=1))
        return np.array(a.dot(b.T).todense()) / np.outer(norm_a, norm_b)

    def search(self, 
              query: str, 
              top_k: int = 10) -> List[Dict[str, Any]]:
        # Get query embeddings
        query_sparse = self._get_sparse_embeddings([query])
        query_dense = self._get_dense_embeddings([query])
        
        # Calculate similarities
        sparse_scores = self._sparse_cosine_similarity(
            query_sparse, 
            self.doc_sparse_embeddings
        )[0]
        
        dense_scores = self._cosine_similarity(
            query_dense, 
            self.doc_dense_embeddings
        )[0]
        
        # Combine scores using weighted sum
        combined_scores = (
            self.lambda_weight * dense_scores + 
            (1 - self.lambda_weight) * sparse_scores
        )
        
        # Get top-k indices and scores
        top_indices = np.argsort(combined_scores)[::-1][:top_k]
        top_scores = combined_scores[top_indices]
        
        results = []
        for idx, score in zip(top_indices, top_scores):
            results.append({
                "index": int(idx),
                "score": float(score),
                "sparse_score": float(sparse_scores[idx]),
                "dense_score": float(dense_scores[idx])
            })
            
        return results

class ClearRetriever:
    # Method in https://arxiv.org/pdf/2004.13969
    def __init__(self, model_path: str, bert_model: str = "bert-base-uncased", device: str = "cuda"):
        self.device = device
        self.model = ResidualDenseRetriever(model_name=bert_model, device=device)
        self.model.load_state_dict(torch.load(model_path, map_location=device))
        self.model.eval()

        # Storage for document embeddings
        self.doc_embeddings = None
        self.documents = []

    def _get_dense_embeddings(self, texts: List[str]) -> np.ndarray:
        """Get dense embeddings for a list of texts using the CLEAR model."""
        embeddings = []
        with torch.no_grad():
            for i in range(0, len(texts), 64):  # Batch processing for efficiency
                batch = texts[i:i + 64]
                embeddings.append(self.model(batch).cpu().numpy())
        return np.vstack(embeddings)

    def index_documents(self, documents: List[str]):
        self.documents = documents
        self.doc_embeddings = self._get_dense_embeddings(documents)
        print(f"Indexed {len(documents)} documents.")

    def search(self, query: str, top_k: int = 10) -> List[Dict[str, Any]]:
        # Compute query embedding
        query_embedding = self._get_dense_embeddings([query])[0]  # Single query embedding

        # Compute cosine similarities
        norm_query = np.linalg.norm(query_embedding)
        norm_docs = np.linalg.norm(self.doc_embeddings, axis=1)
        similarities = np.dot(self.doc_embeddings, query_embedding) / (norm_docs * norm_query)

        # Get top-k results
        top_indices = np.argsort(similarities)[::-1][:top_k]
        top_scores = similarities[top_indices]

        results = []
        for idx, score in zip(top_indices, top_scores):
            results.append({
                "index": int(idx),
                "score": float(score),
                "document": self.documents[idx]
            })

        return results

class ColBERTRetriever:
    def __init__(self, 
                 model_name: str = "colbert-ir/colbertv2.0", 
                 device: str = "cuda" if torch.cuda.is_available() else "cpu",
                 max_length: int = 512):
        """
        Initialize ColBERT retriever for hybrid retrieval.
        """
        self.device = device
        self.max_length = max_length

        # Initialize ColBERT tokenizer and model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(device)

        # Storage for token embeddings and documents
        self.doc_token_embeddings = []
        self.documents = []

    def _get_token_embeddings(self, texts: List[str]) -> List[torch.Tensor]:
        """Get token embeddings for a list of texts."""
        token_embeddings = []
        self.model.eval()
        with torch.no_grad():
            for text in texts:
                inputs = self.tokenizer(
                    text,
                    return_tensors="pt",
                    truncation=True,
                    padding="max_length",
                    max_length=self.max_length
                ).to(self.device)
                outputs = self.model(**inputs)
                token_embeddings.append(outputs.last_hidden_state.squeeze(0).cpu())
        return token_embeddings

    def index_documents(self, documents: List[str]):
        """Index documents to build token-level embeddings."""
        print("Indexing documents for ColBERT...")
        self.documents = documents
        self.doc_token_embeddings = self._get_token_embeddings(documents)
        print(f"Indexed {len(documents)} documents.")

    def search(self, query: str, top_k: int = 10) -> List[Dict[str, Any]]:
        """Retrieve top-k documents for the given query."""
        # Get token embeddings for the query
        query_token_embeddings = self._get_token_embeddings([query])[0]

        # Compute similarity scores
        results = []
        for idx, doc_token_embedding in enumerate(self.doc_token_embeddings):
            # Calculate max-sim for each query token
            similarity = torch.mm(query_token_embeddings, doc_token_embedding.T).max(dim=1).values.sum().item()
            results.append({"index": idx, "score": similarity, "document": self.documents[idx]})

        # Sort by score and return top-k results
        return sorted(results, key=lambda x: x["score"], reverse=True)[:top_k]


def test_hybrid_retrievers():
    # Sample documents
    documents = [
        "The quick brown fox jumps over the lazy dog.",
        "Machine learning is a subset of artificial intelligence.",
        "Python is a popular programming language.",
        "Neural networks are inspired by biological neurons.",
        "Deep learning has revolutionized computer vision."
    ]
    
    query = "What is artificial intelligence?"
    
    # Initialize retrievers
    retrievers = {
        # "ScoreFusion": ScoreFusionRetriever(lambda_weight=0.7),  # 70% dense, 30% sparse
        # "CLEAR": ClearRetriever(model_path="clear_model.pt", bert_model="bert-base-uncased"),
        "ColBERT": ColBERTRetriever()
    }
    
    # Index documents
    for name, retriever in retrievers.items():
        print(f"\nIndexing documents for {name}...")
        retriever.index_documents(documents)
    
    # Test query
    print(f"\nTesting query: '{query}'")
    for name, retriever in retrievers.items():
        print(f"\nResults for {name}:")
        results = retriever.search(query, top_k=3)
        for rank, result in enumerate(results, 1):
            print(f"{rank}. Score: {result['score']:.4f} - Document: {documents[result['index']]}")

if __name__ == "__main__":
    test_hybrid_retrievers()


tokenizer_config.json:   0%|          | 0.00/405 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]


Indexing documents for ColBERT...
Indexing documents for ColBERT...
Indexed 5 documents.

Testing query: 'What is artificial intelligence?'

Results for ColBERT:
1. Score: 54817.6406 - Document: Machine learning is a subset of artificial intelligence.
2. Score: 31132.2148 - Document: Python is a popular programming language.
3. Score: 30283.8242 - Document: Deep learning has revolutionized computer vision.
